# 00 — Data preparation and base feature tables

This notebook loads the event-level Douyin dataset, performs basic validation, and creates the three entity-level base tables used by the later notebooks:

- `user_features.csv` — one row per user
- `author_features.csv` — one row per author
- `video_features.csv` — one row per video

The later notebooks add dashboard metrics, bins, and Power BI sort columns.

## 1. Load libraries and define paths

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "douyin_dataset.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at {RAW_PATH}. "
        "Place douyin_dataset.csv in data/raw/ before running this notebook."
    )

df = pd.read_csv(RAW_PATH)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Rows: 1,737,312
Columns: 14


## 2. Basic cleaning and validation

In [2]:
# Remove a saved CSV index column when present.
if "Unnamed: 0" in df.columns:
    df = df.drop(columns="Unnamed: 0")

required_columns = {
    "uid", "item_id", "author_id", "item_city", "music_id",
    "finish", "like", "duration_time", "real_time", "date"
}
missing_required = sorted(required_columns.difference(df.columns))
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Convert date columns before date-range calculations.
for column in ["real_time", "date"]:
    df[column] = pd.to_datetime(df[column], errors="coerce")

print(f"Duplicate event rows: {df.duplicated().sum():,}")
print("Missing values in required columns:")
display(df[list(sorted(required_columns))].isna().sum())

Duplicate event rows: 0
Missing values in required columns:


author_id        0
date             0
duration_time    0
finish           0
item_city        0
item_id          0
like             0
music_id         0
real_time        0
uid              0
dtype: int64

## 3. Create the user feature table

Each row represents one user. Counts are aggregated from the observed interaction rows.

In [3]:
user_features = (
    df.groupby("uid", as_index=False)
      .agg(
          total_views=("item_id", "size"),
          unique_videos=("item_id", "nunique"),
          unique_authors=("author_id", "nunique"),
          unique_musics=("music_id", "nunique"),
          finish_count=("finish", "sum"),
          like_count=("like", "sum"),
          avg_duration_time=("duration_time", "mean"),
      )
)

user_features["avg_duration_time"] = user_features["avg_duration_time"].round(2)

assert user_features["uid"].is_unique
print(f"Users: {len(user_features):,}")
user_features.head()

Users: 59,232


,uid,total_views,unique_videos,unique_authors,unique_musics,finish_count,like_count,avg_duration_time
0,0,34,34,31,31,18,0,12.06
1,1,28,28,28,26,14,1,12.36
2,2,56,56,56,47,19,0,10.36
3,3,117,117,116,89,60,1,9.98
4,4,123,123,117,94,77,0,10.85


## 4. Create the author feature table

Each row represents one author. `activities` is the number of calendar days between the earliest and latest `date` associated with the author's observed content records. It is a span, not proof that the author posted every day.

In [4]:
author_features = (
    df.groupby("author_id", as_index=False)
      .agg(
          total_views=("item_id", "size"),
          like_count=("like", "sum"),
          finish_count=("finish", "sum"),
          avg_duration_time=("duration_time", "mean"),
          unique_videos=("item_id", "nunique"),
          unique_musics=("music_id", "nunique"),
          unique_real_time=("real_time", "nunique"),
          first_date=("date", "min"),
          last_date=("date", "max"),
      )
)

author_features["activities"] = (
    author_features["last_date"] - author_features["first_date"]
).dt.days + 1

author_features["avg_duration_time"] = author_features["avg_duration_time"].round(2)
author_features = author_features.drop(columns=["first_date", "last_date"])

assert author_features["author_id"].is_unique
print(f"Authors: {len(author_features):,}")
author_features.head()

Authors: 208,187


,author_id,total_views,like_count,finish_count,avg_duration_time,unique_videos,unique_musics,unique_real_time,activities
0,0,1,0,0,10.00,1,1,1,1
1,1,16,0,8,8.88,3,3,3,9
2,3,311,3,203,9.00,1,1,1,1
3,5,1054,33,485,7.04,5,4,5,28
4,8,4,0,3,19.00,1,1,1,1


## 5. Create the video feature table

Each row represents one video. `publish_time` follows the original project logic and uses the first `date` recorded for each `item_id`.

In [5]:
video_features = (
    df.groupby("item_id", as_index=False)
      .agg(
          item_city=("item_city", "first"),
          music_id=("music_id", "first"),
          publish_time=("date", "first"),
          total_views=("uid", "size"),
          like_count=("like", "sum"),
      )
)

assert video_features["item_id"].is_unique
print(f"Videos: {len(video_features):,}")
video_features.head()

Videos: 449,472


,item_id,item_city,music_id,publish_time,total_views,like_count
0,0,24.0,220.0,2019-10-27,24,0
1,1,63.0,574.0,2019-10-27,1309,5
2,3,7.0,26289.0,2019-10-27,2,0
3,4,146.0,162.0,2019-10-27,613,3
4,7,33.0,540.0,2019-10-09,2,0


## 7. Export the base feature tables

In [7]:
output_files = {
    "user_features.csv": user_features,
    "author_features.csv": author_features,
    "video_features.csv": video_features,
}

for filename, table in output_files.items():
    output_path = PROCESSED_DIR / filename
    table.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Exported {len(table):,} rows -> {output_path}")

Exported 59,232 rows -> /home/221eca97-9840-48b7-86b7-6d245b0eaaec/TikTok-Data/data/processed/user_features.csv
Exported 208,187 rows -> /home/221eca97-9840-48b7-86b7-6d245b0eaaec/TikTok-Data/data/processed/author_features.csv
Exported 449,472 rows -> /home/221eca97-9840-48b7-86b7-6d245b0eaaec/TikTok-Data/data/processed/video_features.csv
